<a href="https://colab.research.google.com/github/rist-kobe/HPC-Programming/blob/main/Tuning/sample_code/sample_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Scalar Programming - Sample Code

Setup 後に、各セクションを個別に実行できます。


## Setup


GNU Fortran をインストールします。NVIDIA HPC SDK はオプションとしてコメントアウトしています。


In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential gfortran curl gnupg

# Optional: install NVIDIA HPC SDK (large download).
# Uncomment only when needed.
#!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
#!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
#!sudo apt-get update -y
#!sudo apt-get install -y nvhpc-22-7-cuda-multi


リポジトリを clone して作業ディレクトリへ移動します。


In [ ]:
%cd /content
!rm -rf HPC-Programming
!git clone https://github.com/rist-kobe/HPC-Programming.git
%cd /content/HPC-Programming/Tuning/sample_code
!ls


実行環境の情報（コンパイラ、カーネル、CPU）を表示します。


In [ ]:
!gcc --version
!g++ --version
!gfortran --version
!uname -a
!lscpu


## 01_timer: By-hand Timers and gprof


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/01_timer


README に沿って、Elapsed time / CPU time / gprof を実行します。


Step 1: Elapsed time (default flags) をビルド・実行し、`outfile` を確認します。


In [ ]:
%%bash
cd src/c
make veryclean
make
cd ../../tests/c
bash run.sh
echo "--- outfile ---"
tail -n +1 outfile


Step 2: `-DUSE_CPU_TIMER` に切り替えて再実行します。


In [ ]:
%%bash
cd src/c
python - <<'PY2'
from pathlib import Path
p=Path('Makefile')
t=p.read_text()
t=t.replace('CFLAGS=-g -Wall -O0 -std=gnu99 -DUSE_ELP_TIMER','#CFLAGS=-g -Wall -O0 -std=gnu99 -DUSE_ELP_TIMER',1)
t=t.replace('#CFLAGS=-g -Wall -O0 -std=gnu99 -DUSE_CPU_TIMER','CFLAGS=-g -Wall -O0 -std=gnu99 -DUSE_CPU_TIMER',1)
p.write_text(t)
print('Makefile updated for CPU timer')
PY2
make veryclean
make
cd ../../tests/c
bash run.sh
echo "--- outfile ---"
tail -n +1 outfile


Step 3: `-pg` を有効化し、`run.sh` の gprof 行を有効化してプロファイルします。


In [ ]:
%%bash
cd src/c
python - <<'PY2'
from pathlib import Path
p=Path('Makefile')
t=p.read_text()
t=t.replace('CFLAGS=-g -Wall -O0 -std=gnu99 -DUSE_CPU_TIMER','#CFLAGS=-g -Wall -O0 -std=gnu99 -DUSE_CPU_TIMER',1)
t=t.replace('#CFLAGS=-pg -g -Wall -O0 -std=gnu99','CFLAGS=-pg -g -Wall -O0 -std=gnu99',1)
p.write_text(t)
print('Makefile updated for gprof')
PY2
make veryclean
make
cd ../../tests/c
python - <<'PY2'
from pathlib import Path
p=Path('run.sh')
t=p.read_text()
t=t.replace('#sleep 10s','sleep 10s',1)
t=t.replace('#gprof $EXE > prof.out','gprof $EXE > prof.out',1)
p.write_text(t)
print('run.sh updated for gprof')
PY2
bash run.sh
echo "--- prof.out (head) ---"
head -n 40 prof.out


## 02_timer-res: Check resolution of timer


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/02_timer-res


README の C 例に沿ってビルド・実行します。


Compile: `src/c` で `make`。


In [ ]:
%%bash
cd src/c
make


Run: `tests/c/run.sh` を実行します。


In [ ]:
%%bash
cd tests/c
bash run.sh


## 03_prof-ex: Exercise on a performance analysis with gprof (and perf)


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/03_prof-ex


README の前提どおり、Mersenne Twister のソース配置が必要です。


Preparation: `src/c` に `mt19937ar.c` と `mt19937ar.h` を配置してください。


In [ ]:
%%bash
cd src/c
if [[ -f mt19937ar.c && -f mt19937ar.h ]]; then
  echo "Mersenne Twister sources found."
else
  echo "Missing mt19937ar.c / mt19937ar.h. Please download and place them in src/c first."
fi


Compile: 必要ファイルがある場合に `make`。


In [ ]:
%%bash
cd src/c
if [[ -f mt19937ar.c && -f mt19937ar.h ]]; then
  make
else
  echo "Skip compile until mt19937ar sources are added."
fi


Run + gprof: `tests/c/run.sh`。必要に応じて `tests/perf` も利用します。


In [ ]:
%%bash
cd tests/c
if [[ -x ../../src/c/run.x ]]; then
  bash run.sh
  ls -1
else
  echo "Skip run because ../../src/c/run.x is not built yet."
fi
echo "perf examples are under tests/perf"


## 04_alloc2d: Check address of 2D arrays


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/04_alloc2d


README の C 例に沿って実行し、`outlist` を確認します。


Compile + Run


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- outlist ---"
tail -n +1 outlist


## 05_mattp: Loop blocking: transpose a matrix


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/05_mattp


README の C 例に沿ってビルド・実行し、`outlist` を確認します。


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- outlist ---"
tail -n +1 outlist


## 06_matmatp: Blocking in matrix-matrix products on dense matrices


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/06_matmatp


README の C 例に沿って `src/c` でビルド後、`tests/c` で実行します。


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- generated outputs ---"
ls -1 output.* 2>/dev/null || true


## 07_stripmining: Strip mining


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/07_stripmining


README の C 例に沿ってビルド・実行します（注: README に「通常は非推奨」と記載）。


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh


## 08_thrashing: Thrashing


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/08_thrashing


README の C 例に沿ってビルド・実行し、`output` と `add.info` を確認します。


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- output (tail) ---"
tail -n 20 output
echo "--- add.info (tail) ---"
tail -n 20 add.info


## 09_unroll: Loop unrolling: on outer loop


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/09_unroll


README の C 例に沿ってビルド・実行し、`outlist` を確認します。


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- outlist ---"
tail -n +1 outlist


## 10_simd-add: Example of SIMD: Vector addition


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/10_simd-add


README の GNU 例どおり `make all` を使い、`run.sh` 実行後にログを確認します。


In [ ]:
%%bash
cd src/c
make all
cd ../../tests/c
bash run.sh
echo "--- run_v.log (tail) ---"
tail -n 20 run_v.log
echo "--- run_nv.log (tail) ---"
tail -n 20 run_nv.log


## 11_simd-nsimple: Example of SIMD: Non-simple loops


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/11_simd-nsimple


README の GNU 例どおり `make all` と `run.sh` を実行します。


In [ ]:
%%bash
cd src/c
make all
cd ../../tests/c
bash run.sh
echo "--- run_v.log (tail) ---"
tail -n 20 run_v.log
echo "--- run_nv.log (tail) ---"
tail -n 20 run_nv.log


## 12_swp: Example of software pipelining


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/12_swp


README の C 例に沿ってビルド・実行し、`outfile.*` を確認します。


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- generated outputs ---"
ls -1 outfile.* 2>/dev/null || true


## 13_polynomial: Comparison between different implementations of a compute-bound kernel, n-th order polynomial


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/13_polynomial


README に沿って複数実装を比較できるよう、全サブディレクトリで compile/run します。


In [ ]:
%%bash
for d in c cpp cpp.et14 cpp.etp cpp.oo cpp.valarray f90 f90.forall f08.concurrent; do
  echo "=== build src/$d ==="
  (cd src/$d && make) || { echo "[WARN] build failed in src/$d"; continue; }
  echo "=== run tests/$d/run.sh ==="
  (cd tests/$d && bash run.sh) || echo "[WARN] run failed in tests/$d"
done
echo "--- sample logs ---"
for d in c cpp cpp.oo f90; do
  echo "[$d]"
  tail -n 5 tests/$d/run.log || true
done


## 14_endian: Check endianness


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/14_endian


README の C 例に沿ってビルド・実行し、`logfile` を確認します。


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- logfile ---"
tail -n +1 logfile
echo "--- lscpu endian check ---"
lscpu | grep -i -E "endian|byte" || true


## 15_io-format: Formatted (text) vs. Binary outputs


In [ ]:
%cd /content/HPC-Programming/Tuning/sample_code/15_io-format


README の C 例に沿ってビルド・実行し、`logfile` を確認します。


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- logfile ---"
tail -n +1 logfile
